# Generating shrunken logFC (posterior-mean) matrices

Each perturbation gets several logFC estimates — one from all its cells, plus one per
subsample — in a single matrix per context, with the cell count behind every row recorded.

Perturbations with more than `MIN_CELLS_TO_SUBSAMPLE` cells are subsampled at sizes
spanning `[MIN_CELLS_TO_SUBSAMPLE, n_cells)`; the cells in each are drawn at random. A
fixed set of `N_CONTROL_CELLS` controls is shared by every row.

Each context runs in its own subprocess, so its memory is reclaimed on exit — these
objects are ~150 GB resident and Python does not reliably hand memory back. For long runs
prefer the command line, which does the same thing without a kernel to lose:

```bash
python scripts/build_posterior_matrices.py \
    --screen-dir /processed_datasets/VCI/ChemoGenetic_H1_Basak \
    --out-dir /large_storage/ctc/beraslan/ModuleFinder/posterior_matrices \
    --n-subsamples 6 --summary summary.csv
```

Requires R with the `ashr` package.

In [ ]:
import sys
from pathlib import Path

PROJECT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT / "src"))

from module_finder.de import run_screen

## Parameters

In [ ]:
SCREEN_DIR = Path("/processed_datasets/VCI/ChemoGenetic_H1_Basak")
OUT_DIR = Path("/large_storage/ctc/beraslan/ModuleFinder/posterior_matrices")

# Listed rather than globbed: SCREEN_DIR also holds Round-1 contexts
# (CHIR, DMSO, KYA, LDN, PFI1, RGFP) with the same file layout.
CONTEXTS = [
    "DMSO_round2", "DMSO_round2_batch2",
    "AR-A014418", "AZD4573", "Bisindolylmaleimide-I", "CHIR-98014", "DG-172",
    "JTE-607", "LDN-193189", "LY2090314", "Lexibulin", "NSC95397", "PP121",
    "Romidepsin", "Stattic", "VX-11e",
]

PERTURBATION_KEY = "target_gene"
CONTROL_LABEL = "non-targeting"
LAYER = None                    # None uses .X

N_SUBSAMPLES = 6                # subsamples per eligible perturbation
SPACING = "linear"              # or "log" (covers precision more evenly)
MIN_CELLS_TO_SUBSAMPLE = 50     # only subsample perturbations with more cells than this
N_CONTROL_CELLS = 10_000        # fixed control set, shared by every row
MIN_CELLS = 20                  # ComputeSE.py drops perturbations below this
CHUNK_PERTS = 100               # labels per ashr fit
N_ASHR_JOBS = 16                # parallel R workers

## Run

Resumable: contexts whose matrix already exists are skipped, and a failure in one does
not stop the rest. Start with `CONTEXTS[:1]` to get a real timing before doing all 16.

In [ ]:
summary = run_screen(
    screen_dir=SCREEN_DIR,
    contexts=CONTEXTS,
    out_dir=OUT_DIR,
    perturbation_key=PERTURBATION_KEY,
    control_label=CONTROL_LABEL,
    layer=LAYER,
    n_subsamples=N_SUBSAMPLES,
    spacing=SPACING,
    min_cells_to_subsample=MIN_CELLS_TO_SUBSAMPLE,
    n_control_cells=N_CONTROL_CELLS,
    min_cells=MIN_CELLS,
    chunk_perts=CHUNK_PERTS,
    n_ashr_jobs=N_ASHR_JOBS,
    seed=0,
)
summary

## Results

Per context: `PosteriorMean_matrix_<context>.parquet` (labels × genes, `<pert>` rows are
full-data estimates and `<pert>__sub<k>` rows are subsamples), `row_metadata.csv` (the
cell count behind each row) and `module_finder_manifest.json`.

In [ ]:
import pandas as pd

context = CONTEXTS[0]
matrix = pd.read_parquet(OUT_DIR / context / f"PosteriorMean_matrix_{context}.parquet")
rows = pd.read_csv(OUT_DIR / context / "row_metadata.csv")

print(matrix.shape)
display(matrix.iloc[:6, :5])
rows.head(10)